In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import pickle

"""
letras_a_numeros = {
    'A' : 0,
    'B': 1,
    'C': 2,
    'D': 3,
    'E': 4,
    'F': 5,
    'G': 6,
    'H': 7,
    'I': 8,
    'J': 9,
    'K': 10,
    'L': 11,
    'M': 12,
    'N': 13,
    'O': 14,
    'P': 15,
    'Q': 16,
    'R': 17,
    'S': 18,
    'T': 19,
    'U': 20,
    'V': 21,
    'W': 22,
    'X': 23,
    'Y': 24,
    'Z': 25,
    'CH': 26,
    'Ñ': 27,
    'LL' : 28
} """

# 1. CARGAR LOS DATOS
# -------------------
# Leemos los CSVs indicando que NO tienen cabecera (header=None)
df_Adrian = pd.read_csv('gestos_Adrian.csv', header=None, sep=',')
df_Adrian_CHÑ = pd.read_csv('gestos_CHÑ_Adrian.csv', header=None, sep=',')

df_Alex = pd.read_csv('gestos_Alejandro.csv', header=None, sep=',')
df_Alex_CHÑ = pd.read_csv('gestos_CHÑ_Alejandro.csv', header=None, sep=',')

df_Markos = pd.read_csv('gestos_Markos.csv', header=None, sep=',')



Ajustamos la etiqueta de la CH y la Ñ


In [ ]:
df_Adrian_CHÑ[0] = df_Adrian_CHÑ[0] + 26
df_Alex_CHÑ[0] = df_Alex_CHÑ[0] + 26

In [ ]:
# Juntamos todo en un solo gran dataset
dataset = pd.concat([df_Adrian_CHÑ, df_Adrian, df_Alex_CHÑ, df_Alex, df_Markos], ignore_index=True)

dataset_ordenado = dataset.sort_values(by=0)
dataset_ordenado = dataset_ordenado.reset_index(drop = True)
dataset_ordenado

,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,40,41,42
0,0,0.0,0.0,-0.125434,-0.582747,-0.373911,-0.925086,-0.669318,-0.883231,-0.796052,...,-0.738987,-0.048271,-0.766622,0.370168,-0.932246,0.237634,-0.846601,0.188929,-0.741274,0.211599
1,0,0.0,0.0,-0.129341,-0.575150,-0.390550,-0.923389,-0.693302,-0.899398,-0.807739,...,-0.753423,-0.060856,-0.792866,0.373833,-0.955993,0.232790,-0.872008,0.184200,-0.770254,0.210200
2,0,0.0,0.0,-0.110259,-0.591734,-0.370281,-0.958609,-0.684183,-0.944134,-0.794823,...,-0.749971,-0.077790,-0.781303,0.359822,-0.954979,0.213712,-0.870549,0.166692,-0.766885,0.196679
3,0,0.0,0.0,-0.118498,-0.578546,-0.372392,-0.929917,-0.677990,-0.905846,-0.786684,...,-0.744333,-0.048738,-0.757551,0.352881,-0.939547,0.226566,-0.851580,0.190154,-0.750551,0.218640
4,0,0.0,0.0,-0.115145,-0.564735,-0.374121,-0.945759,-0.689855,-0.947720,-0.796995,...,-0.741553,-0.047644,-0.762245,0.379595,-0.946575,0.239598,-0.860092,0.204263,-0.755314,0.236854
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13934,27,0.0,0.0,-0.110407,0.130312,-0.140952,0.319343,-0.093868,0.472581,-0.023494,...,0.038860,0.448105,0.132411,0.310082,0.125050,0.482928,0.093730,0.438323,0.079494,0.375562
13935,27,0.0,0.0,-0.112702,0.125226,-0.143661,0.316258,-0.095362,0.470282,-0.022761,...,0.038900,0.446807,0.130773,0.308858,0.120514,0.486429,0.089666,0.439497,0.076702,0.375428
13936,27,0.0,0.0,-0.113276,0.128783,-0.144481,0.320870,-0.097687,0.477301,-0.024666,...,0.036005,0.445757,0.134172,0.309829,0.123246,0.487470,0.092011,0.440772,0.078679,0.376634
13937,27,0.0,0.0,-0.112615,0.125953,-0.145142,0.317207,-0.099731,0.474216,-0.027343,...,0.035134,0.447975,0.130842,0.309812,0.122308,0.488488,0.092456,0.442109,0.078885,0.378366


In [ ]:
dataset_ordenado[0].value_counts()

,count
0,
0,770
5,666
7,653
4,645
3,629
15,627
6,618
16,614
19,600


In [ ]:

# 2. PREPARAR DATOS (X e y)
# -------------------------
# La columna 0 es la etiqueta (A, B, C... o Nada)
y = dataset_ordenado.iloc[:, 0] # Ensure y is of integer type
# Del 1 al final son las coordenadas de la mano
X = dataset_ordenado.iloc[:, 1:]

X

,1,2,3,4,5,6,7,8,9,10,...,33,34,35,36,37,38,39,40,41,42
0,0.0,0.0,-0.125434,-0.582747,-0.373911,-0.925086,-0.669318,-0.883231,-0.796052,-0.677752,...,-0.738987,-0.048271,-0.766622,0.370168,-0.932246,0.237634,-0.846601,0.188929,-0.741274,0.211599
1,0.0,0.0,-0.129341,-0.575150,-0.390550,-0.923389,-0.693302,-0.899398,-0.807739,-0.670190,...,-0.753423,-0.060856,-0.792866,0.373833,-0.955993,0.232790,-0.872008,0.184200,-0.770254,0.210200
2,0.0,0.0,-0.110259,-0.591734,-0.370281,-0.958609,-0.684183,-0.944134,-0.794823,-0.710101,...,-0.749971,-0.077790,-0.781303,0.359822,-0.954979,0.213712,-0.870549,0.166692,-0.766885,0.196679
3,0.0,0.0,-0.118498,-0.578546,-0.372392,-0.929917,-0.677990,-0.905846,-0.786684,-0.681413,...,-0.744333,-0.048738,-0.757551,0.352881,-0.939547,0.226566,-0.851580,0.190154,-0.750551,0.218640
4,0.0,0.0,-0.115145,-0.564735,-0.374121,-0.945759,-0.689855,-0.947720,-0.796995,-0.712290,...,-0.741553,-0.047644,-0.762245,0.379595,-0.946575,0.239598,-0.860092,0.204263,-0.755314,0.236854
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13934,0.0,0.0,-0.110407,0.130312,-0.140952,0.319343,-0.093868,0.472581,-0.023494,0.557736,...,0.038860,0.448105,0.132411,0.310082,0.125050,0.482928,0.093730,0.438323,0.079494,0.375562
13935,0.0,0.0,-0.112702,0.125226,-0.143661,0.316258,-0.095362,0.470282,-0.022761,0.554909,...,0.038900,0.446807,0.130773,0.308858,0.120514,0.486429,0.089666,0.439497,0.076702,0.375428
13936,0.0,0.0,-0.113276,0.128783,-0.144481,0.320870,-0.097687,0.477301,-0.024666,0.562549,...,0.036005,0.445757,0.134172,0.309829,0.123246,0.487470,0.092011,0.440772,0.078679,0.376634
13937,0.0,0.0,-0.112615,0.125953,-0.145142,0.317207,-0.099731,0.474216,-0.027343,0.562369,...,0.035134,0.447975,0.130842,0.309812,0.122308,0.488488,0.092456,0.442109,0.078885,0.378366


In [ ]:

# Dividimos: 80% para estudiar (Train), 20% para el examen (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:

# 3. ENTRENAR EL MODELO (Random Forest)
# -------------------------------------
print(">>> Entrenando modelo...")
model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
model.fit(X_train, y_train)



>>> Entrenando modelo...


RandomForestClassifier(max_depth=15, random_state=42)

In [ ]:
# 4. EVALUACIÓN (EL EXAMEN)
# -------------------------
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("-" * 30)
print(f"PRECISIÓN DEL MODELO: {accuracy * 100:.2f}%")
print("-" * 30)
print("Informe detallado por letra:")
print(classification_report(y_test, y_pred))


------------------------------
PRECISIÓN DEL MODELO: 99.96%
------------------------------
Informe detallado por letra:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       164
           1       1.00      1.00      1.00       101
           2       1.00      1.00      1.00       115
           3       1.00      1.00      1.00       125
           4       1.00      1.00      1.00       129
           5       1.00      1.00      1.00       133
           6       1.00      1.00      1.00       138
           7       1.00      1.00      1.00       118
           8       1.00      1.00      1.00       107
           9       1.00      1.00      1.00       111
          10       1.00      1.00      1.00       113
          11       1.00      1.00      1.00       128
          12       1.00      1.00      1.00       126
          13       1.00      1.00      1.00       113
          14       1.00      1.00      1.00       123
          15   

In [ ]:
# 5. GUARDAR EL MODELO
# --------------------
#joblib.dump(model, 'randomforestEstaticas_model.pkl')
with open('./randomforestEstaticas_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("¡Modelo guardado como 'randomforestEstaticas_model.pkl'!")

¡Modelo guardado como 'randomforestEstaticas_model.pkl'!
